
# NYC Taxi → product_ec  
## Notebook d’exploration & calibration (référence)

### Objectif
Ce notebook sert de support d’exploration et de calibration pour le détournement  
du dataset **NYC Taxi** vers une table métier `product_ec`.

Il permet de :
- comprendre les ordres de grandeur des données sources
- justifier les règles de transformation retenues
- calibrer les empreintes carbone (process / transport)
- conserver une trace claire des choix métier


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np



## Chargement des données


In [ ]:

DATA_PATH = Path("Data-sources-nawres/nyc_taxi/data/yellow_tripdata_2025-01.parquet")

df = pd.read_parquet(DATA_PATH)
df.shape



## Aperçu des données


In [ ]:

df.columns


In [ ]:

df.head()



## Colonnes utilisées pour le détournement

- `PULocationID` → entrepôts  
- `DOLocationID` → produits  
- `fare_amount` → empreinte process  
- `trip_distance` → empreinte transport  
- `total_amount` → proxy de stock  

Les autres colonnes sont ignorées.



## Analyse statistique des colonnes clés


In [ ]:

df[["fare_amount", "trip_distance"]].describe(
    percentiles=[0.5, 0.9, 0.95, 0.99]
)


In [ ]:

(df["fare_amount"] <= 0).mean(), (df["trip_distance"] <= 0).mean()


In [ ]:

df["fare_amount"].sort_values(ascending=False).head(10)


In [ ]:

df["trip_distance"].sort_values(ascending=False).head(10)



## Nettoyage (clipping des outliers)


In [ ]:

fare = df["fare_amount"].clip(
    lower=0,
    upper=df["fare_amount"].quantile(0.99)
)

dist = df["trip_distance"].clip(
    lower=0,
    upper=df["trip_distance"].quantile(0.99)
)



## Référentiel métier – Textile (rappel)

- Empreinte Process : 2 → 50 kg CO2eq  
- Empreinte Transport : 0.05 → 8 kg CO2eq  

Règle :
- hors transport aérien,  
  **ec_process ≈ 15–20 × ec_transport**



## Calibration de l’empreinte Process


In [ ]:

ec_process = 2 + 48 * (
    (fare - fare.min()) / (fare.max() - fare.min())
)

ec_process.describe(percentiles=[0.5, 0.9, 0.95])



## Calibration de l’empreinte Transport


In [ ]:

ec_transport = 0.05 + 7.95 * (
    (dist - dist.min()) / (dist.max() - dist.min())
)

ec_transport.describe(percentiles=[0.5, 0.9, 0.95])



## Vérification du ratio Process / Transport


In [ ]:

(ec_process / ec_transport).describe(percentiles=[0.5, 0.9, 0.95])



## Prototype minimal de détournement


In [ ]:

N_PRODUCT_REF = 50_000
N_WAREHOUSE = 200

sample = df.sample(n=200_000, random_state=42).copy()

sample["id_warehouse"] = (sample["PULocationID"] % N_WAREHOUSE) + 1
sample["id_product_ref"] = (sample["DOLocationID"] % N_PRODUCT_REF) + 1

sample["ec_process"] = ec_process.loc[sample.index]
sample["ec_transport"] = ec_transport.loc[sample.index]
sample["ec_total"] = sample["ec_process"] + sample["ec_transport"]

sample["stock_qty"] = (
    (sample["total_amount"].clip(lower=0) /
     sample["total_amount"].quantile(0.95) * 199)
    .clip(1, 200)
    .round()
    .astype("int")
)

sample["id_product_ec"] = np.arange(1, len(sample) + 1)


In [ ]:

sample[
    ["id_product_ec", "id_product_ref", "id_warehouse",
     "stock_qty", "ec_process", "ec_transport", "ec_total"]
].head(10)



## Conclusion

Ce notebook valide :
- les ordres de grandeur des données sources
- la cohérence des règles de détournement
- la compatibilité avec un référentiel métier réel

Il sert de référence avant l’implémentation Spark.
